In [1]:
import numpy as np

from model_ranking.NCTI import NCTI_Score
from model_ranking.feature_ranking import get_precomputed_feature_path
from model_ranking.utils import load_h5

In [2]:
models_hub = ["E_model_Res1", "Hm_model_Res1", "Rm_model_Res1", "V_model_Res1"]
target = "EPFL"
feature_base_path = "/scratch/talks/sampled_features/semantic_segmentation/mitochondria"
key = "decoders.3"


In [6]:
score_dict = {}
for model in models_hub:
    feature_path = get_precomputed_feature_path(model, target, feature_base_path)
    features = load_h5(feature_path, f"{key}_features")
    labels = load_h5(feature_path, f"{key}_labels")

    non_zero_patch_ids = np.where(~np.all(labels == 0, axis=1))[0]
    features = features[non_zero_patch_ids]
    labels = labels[non_zero_patch_ids]

    features_flat = features.reshape(-1, features.shape[-1])
    labels_flat = labels.reshape(-1).astype(np.uint8)
    score_dict[model] = NCTI_Score(features_flat, labels_flat)

1.0
0.547247191011236
(2, 1)
all feat nuc: 844.171128455334
class res nuc: 871.2892150878906
pred: 0.5049052844101124
1.0
0.5477387640449438
(2, 1)
all feat nuc: 844.0268035339244
class res nuc: 866.0457763671875
pred: 0.5047386762640449
1.0
0.5474058988764045
(2, 1)
all feat nuc: 844.0280165211416
class res nuc: 866.0508117675781
pred: 0.5047401246488764
1.0
0.5509929775280898
(2, 1)
all feat nuc: 845.3725932814932
class res nuc: 897.8055725097656
pred: 0.5063279933286516


In [7]:
score_dict

{'E_model_Res1': (844.171128455334, 0.5049052844101124, 6.769973971205728),
 'Hm_model_Res1': (844.0268035339244, 0.5047386762640449, 6.763937766711824),
 'Rm_model_Res1': (844.0280165211416, 0.5047401246488764, 6.763943580937688),
 'V_model_Res1': (845.3725932814932, 0.5063279933286516, 6.799953533178386)}

In [ ]:
all_score = []
cls_score = []
cls_compact = []

for model in models_hub:
    all_score.append(score_dict[model][0])
    cls_score.append(score_dict[model][1])
    cls_compact.append(score_dict[model][2])
    
all_score = np.array(all_score)
cls_score = np.array(cls_score)
cls_compact = np.array(cls_compact)

all_score_min = all_score.min()
all_score_div = all_score.max() - all_score.min()

cls_score_min = cls_score.min()
cls_score_div = cls_score.max() - cls_score.min()

cls_compact_min = cls_compact.min()
cls_compact_div = cls_compact.max() - cls_compact_min

for model in models_hub:
    print(model)
    mascore = (score_dict[model][0] - all_score_min)/all_score_div
    mcscore = (score_dict[model][1] - cls_score_min)/cls_score_div
    cpscore = (score_dict[model][2] - cls_compact_min)/cls_compact_div
    print(mascore)
    print(mcscore)
    print(cpscore)
    score_dict[model] = mcscore  + mascore - cpscore
print("seli")

# print(((all_score - all_score_min)/all_score_div).var())
print(all_score)
print("ncc")
        
# print(((cls_score- cls_score_min)/cls_score_div).var())
print(cls_score)
print("vc")
# print(((cls_compact- cls_compact_min)/cls_compact_div).var())
print(cls_compact)

E_model_Res1
0.10724180479927133
0.1048300240259039
0.16759894585355198
Hm_model_Res1
0.0
0.0
0.0
Rm_model_Res1
0.0009013200014073189
0.0009113252879301195
0.00016143557210473875
V_model_Res1
1.0
1.0
1.0
seli
[844.17112846 844.02680353 844.02801652 845.37259328]
ncc
[0.50490528 0.50473868 0.50474012 0.50632799]
vc
[6.76997397 6.76393777 6.76394358 6.79995353]


In [9]:
score_dict

{'E_model_Res1': 0.37967077467872723,
 'Hm_model_Res1': 0.0,
 'Rm_model_Res1': 0.001974080861442177,
 'V_model_Res1': 3.0}